# Word Frequency Analysis

**Authors:** Benedikt Prisett & Stijn Diemel

This notebook explores word frequency and TF-IDF based approaches for comparing word distributions between characters (Questions 3 & 4). These designs were ultimately not used in the final visualization but document the exploration process.


## Setup


In [1]:
import pandas as pd
import altair as alt

In [2]:
df = pd.read_csv("../data/clean_data/simpsons_script_lines_clean.csv")
alt.data_transformers.enable("vegafusion")
df.head(10).style.set_properties(subset=["spoken_words"], **{"white-space": "pre-wrap"})

,episode_id,season,number_in_season,title,imdb_rating,line_number,character,spoken_words,word_count,sentence_count
0,1,1,1,Simpsons Roasting on an Open Fire,8.200000,2,Marge Simpson,"Ooo, careful, Homer.",3,1
1,1,1,1,Simpsons Roasting on an Open Fire,8.200000,3,Homer Simpson,There's no time to be careful.,6,1
2,1,1,1,Simpsons Roasting on an Open Fire,8.200000,4,Homer Simpson,We're late.,2,1
3,1,1,1,Simpsons Roasting on an Open Fire,8.200000,7,Marge Simpson,"Sorry, Excuse us. Pardon me...",5,2
4,1,1,1,Simpsons Roasting on an Open Fire,8.200000,8,Homer Simpson,"Hey, Norman. How's it going? So you got dragged down here, too... heh, heh. How ya doing, Fred? Excuse me, Fred.",21,6
5,1,1,1,Simpsons Roasting on an Open Fire,8.200000,9,Homer Simpson,Pardon my galoshes.,3,1
6,1,1,1,Simpsons Roasting on an Open Fire,8.200000,10,Seymour Skinner,"Wasn't that wonderful? And now, ""Santas of Many Lands,"" as presented by the entire second grade class.",17,2
7,1,1,1,Simpsons Roasting on an Open Fire,8.200000,11,Marge Simpson,Oh... Lisa's class.,3,2
8,1,1,1,Simpsons Roasting on an Open Fire,8.200000,12,JANEY,"Frohlich weihnachten -- that's German for Merry Christmas. In Germany, Santa's servant Ruprecht gives presents to good children and whipping rods to the parents of bad ones.",27,2
9,1,1,1,Simpsons Roasting on an Open Fire,8.200000,13,Todd Flanders,"Meri Kurimasu. I am Hotseiosha, a Japanese priest who acts like Santa Claus. I have eyes in the back of my head so children better behave when I'm nearby.",29,3


---

## Preprocessing: Tokenization & Word Frequency Tables


In [3]:
import re
from collections import Counter

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))


def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())


# Build word frequencies per character, season, and episode
word_rows = []
for _, row in df.iterrows():
    tokens = [
        w for w in tokenize(row["spoken_words"]) if w not in stop_words and len(w) > 1
    ]
    for word, count in Counter(tokens).items():
        word_rows.append(
            {
                "character": row["character"],
                "season": row["season"],
                "episode_id": row["episode_id"],
                "word": word,
                "count": count,
            }
        )

word_freq_df = pd.DataFrame(word_rows)
print(f"Word frequency table: {len(word_freq_df):,} rows")
word_freq_df.head(10)

Word frequency table: 659,156 rows


,character,season,episode_id,word,count
0,Marge Simpson,1,1,ooo,1
1,Marge Simpson,1,1,careful,1
2,Marge Simpson,1,1,homer,1
3,Homer Simpson,1,1,time,1
4,Homer Simpson,1,1,careful,1
5,Homer Simpson,1,1,late,1
6,Marge Simpson,1,1,sorry,1
7,Marge Simpson,1,1,excuse,1
8,Marge Simpson,1,1,us,1
9,Marge Simpson,1,1,pardon,1


---

## Question 3: Compare the word distribution for a pair of characters, for a selected season

### Design Iterations


In [4]:
# Q3 - iteration 1.1: side-by-side lollipop charts (plain, no highlight)
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
n_words = 20


def get_top_words(character, season, n_words):
    return (
        word_freq_df[
            (word_freq_df["character"] == character)
            & (word_freq_df["season"] == season)
        ]
        .groupby("word", as_index=False)["count"]
        .sum()
        .nlargest(n_words, "count")
    )


top_a = get_top_words(character_a, season, n_words)
top_b = get_top_words(character_b, season, n_words)


def make_lollipop_plain(data, character, color="#4c78a8"):
    base = alt.Chart(data).encode(
        x=alt.X("count:Q", title="Word Count"),
        y=alt.Y("word:N", sort="-x", title="Word"),
        tooltip=["word", "count"],
    )

    points = base.mark_circle(size=80, color=color)
    lines = base.mark_rule(color=color)

    return (lines + points).properties(title=character, width=300, height=400)


chart_a = make_lollipop_plain(top_a, character_a)
chart_b = make_lollipop_plain(top_b, character_b)

(chart_a | chart_b).properties(
    title=alt.TitleParams(
        text=f"Top {n_words} Words per Character \u2014 Season {season}",
        anchor="middle",
    )
)

alt.HConcatChart(...)

In [5]:
# Q3 - iteration 1.2: side-by-side lollipop charts with shared words highlighted
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
n_words = 20

top_a = get_top_words(character_a, season, n_words)
top_b = get_top_words(character_b, season, n_words)

# Find words that appear in both top lists
shared_words = set(top_a["word"]) & set(top_b["word"])
top_a = top_a.assign(
    shared=top_a["word"].apply(lambda w: "Shared" if w in shared_words else "Unique")
)
top_b = top_b.assign(
    shared=top_b["word"].apply(lambda w: "Shared" if w in shared_words else "Unique")
)


def make_lollipop_highlighted(data, character):
    base = alt.Chart(data).encode(
        x=alt.X("count:Q", title="Word Count"),
        y=alt.Y("word:N", sort="-x", title="Word"),
        color=alt.Color(
            "shared:N",
            scale=alt.Scale(domain=["Unique", "Shared"], range=["#4c78a8", "#e45756"]),
            legend=alt.Legend(title="Shared Word"),
        ),
        tooltip=["word", "count", "shared"],
    )

    points = base.mark_circle(size=80)
    lines = base.mark_rule()

    return (lines + points).properties(title=character, width=300, height=400)


chart_a = make_lollipop_highlighted(top_a, character_a)
chart_b = make_lollipop_highlighted(top_b, character_b)

(chart_a | chart_b).properties(
    title=alt.TitleParams(
        text=f"Top {n_words} Words per Character \u2014 Season {season}",
        subtitle=f"{len(shared_words)} words appear in both top lists (highlighted in red)",
        anchor="middle",
    )
)

alt.HConcatChart(...)

In [6]:
# Q3 - iteration 1.3: slope graph comparing word ranks between two characters
# Each line connects the same word's rank in character A (left) vs character B (right)
# Words only in one character's top list are shown as dots without connecting lines
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
n_words = 15

top_a = get_top_words(character_a, season, n_words)
top_b = get_top_words(character_b, season, n_words)

# Assign ranks (1 = most frequent)
top_a = top_a.assign(rank=range(1, len(top_a) + 1))
top_b = top_b.assign(rank=range(1, len(top_b) + 1))

# Force visual left->right order
character_order = [character_a, character_b]
left_character, right_character = character_order

# Get all words that appear in either list
all_words = set(top_a["word"]) | set(top_b["word"])
shared_words = set(top_a["word"]) & set(top_b["word"])

# Build slope data: one row per word per character
slope_rows = []
for word in all_words:
    rank_a = top_a.loc[top_a["word"] == word, "rank"]
    rank_b = top_b.loc[top_b["word"] == word, "rank"]

    if len(rank_a) > 0:
        slope_rows.append(
            {
                "word": word,
                "character": character_a,
                "rank": rank_a.values[0],
                "shared": word in shared_words,
            }
        )
    if len(rank_b) > 0:
        slope_rows.append(
            {
                "word": word,
                "character": character_b,
                "rank": rank_b.values[0],
                "shared": word in shared_words,
            }
        )

slope_df = pd.DataFrame(slope_rows)

# Lines only for shared words
shared_df = slope_df[slope_df["shared"]]
unique_df = slope_df[~slope_df["shared"]]

lines = (
    alt.Chart(shared_df)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X(
            "character:N",
            sort=character_order,
            title=None,
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y(
            "rank:Q",
            title="Rank (1 = most used)",
            sort="descending",
            axis=alt.Axis(grid=False),
        ),
        detail="word:N",
        color=alt.value("#e45756"),
        tooltip=["word", "rank", "character"],
    )
)

# Split data for side-specific label placement
shared_left = shared_df[shared_df["character"] == left_character]
shared_right = shared_df[shared_df["character"] == right_character]
unique_left = unique_df[unique_df["character"] == left_character]
unique_right = unique_df[unique_df["character"] == right_character]

# Labels for shared words (outside)
shared_labels_left = (
    alt.Chart(shared_left)
    .mark_text(fontSize=10, dx=-15, align="right")
    .encode(
        x=alt.X("character:N", sort=character_order),
        y=alt.Y("rank:Q", sort="descending", axis=alt.Axis(grid=False)),
        text="word:N",
    )
)

shared_labels_right = (
    alt.Chart(shared_right)
    .mark_text(fontSize=10, dx=15, align="left")
    .encode(
        x=alt.X("character:N", sort=character_order),
        y=alt.Y("rank:Q", sort="descending", axis=alt.Axis(grid=False)),
        text="word:N",
    )
)

shared_labels = shared_labels_left + shared_labels_right

# Dots for unique words (no connecting line)
unique_dots = (
    alt.Chart(unique_df)
    .mark_circle(size=50, color="#999999")
    .encode(
        x=alt.X("character:N", sort=character_order),
        y=alt.Y("rank:Q", sort="descending", axis=alt.Axis(grid=False)),
        tooltip=["word", "rank", "character"],
    )
)

# Labels for unique words (outside)
unique_labels_left = (
    alt.Chart(unique_left)
    .mark_text(fontSize=9, dx=-15, align="right", color="black", opacity=1)
    .encode(
        x=alt.X("character:N", sort=character_order),
        y=alt.Y("rank:Q", sort="descending", axis=alt.Axis(grid=False)),
        text="word:N",
    )
)

unique_labels_right = (
    alt.Chart(unique_right)
    .mark_text(fontSize=9, dx=15, align="left", color="black", opacity=1)
    .encode(
        x=alt.X("character:N", sort=character_order),
        y=alt.Y("rank:Q", sort="descending", axis=alt.Axis(grid=False)),
        text="word:N",
    )
)

unique_labels = unique_labels_left + unique_labels_right

(lines + shared_labels + unique_dots + unique_labels).properties(
    title=alt.TitleParams(
        text=f"Word Rank Comparison \u2014 {character_a} vs {character_b}, Season {season}",
        subtitle="Red lines = shared words; grey dots = unique to one character",
    ),
    width=300,
    height=500,
)

alt.LayerChart(...)

---

## TF-IDF Approach


In [7]:
# Q3 - TF-IDF computation
# Treats each character in a season as a "document".
# TF = word count / total words for that character in that season
# IDF = log(num_characters / num_characters_using_word) within the season
# Words common to all characters get low IDF; distinctive words get high scores.
import numpy as np


def compute_tfidf(word_freq_df, characters, season):
    """Compute TF-IDF scores for given characters in a specific season."""
    season_df = word_freq_df[word_freq_df["season"] == season]

    # Total words spoken by each character this season (used for TF normalization)
    char_totals = season_df.groupby("character")["count"].sum()

    # How many characters in this season (total number of "documents")
    n_characters = season_df["character"].nunique()

    # For each word, how many characters use it (document frequency)
    word_doc_freq = season_df.groupby("word")["character"].nunique()

    tfidf_rows = []
    for character in characters:
        char_words = (
            season_df[season_df["character"] == character]
            .groupby("word", as_index=False)["count"]
            .sum()
        )
        total = char_totals.get(character, 1)
        for _, row in char_words.iterrows():
            tf = row["count"] / total  # Term frequency (normalized)
            idf = np.log(
                n_characters / word_doc_freq[row["word"]]
            )  # Inverse document frequency
            tfidf_rows.append(
                {"character": character, "word": row["word"], "tfidf": tf * idf}
            )

    return pd.DataFrame(tfidf_rows)

In [8]:
# Q3 - iteration 2: TF-IDF weighted comparison (plain, no highlight)
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
n_words = 20

tfidf_df = compute_tfidf(word_freq_df, [character_a, character_b], season)


def make_tfidf_lollipop_plain(data, character, color="#4c78a8"):
    top_words = data[data["character"] == character].nlargest(n_words, "tfidf")

    base = alt.Chart(top_words).encode(
        x=alt.X("tfidf:Q", title="TF-IDF Score"),
        y=alt.Y("word:N", sort="-x", title="Word"),
        tooltip=["word", alt.Tooltip("tfidf:Q", format=".4f")],
    )

    points = base.mark_circle(size=80, color=color)
    lines = base.mark_rule(color=color)

    return (lines + points).properties(title=character, width=300, height=400)


chart_a = make_tfidf_lollipop_plain(tfidf_df, character_a)
chart_b = make_tfidf_lollipop_plain(tfidf_df, character_b)

(chart_a | chart_b).properties(
    title=alt.TitleParams(
        text=f"Most Distinctive Words (TF-IDF) \u2014 Season {season}",
        anchor="middle",
    )
)

alt.HConcatChart(...)